In [ ]:
!pip install diffusers transformers accelerate safetensors gradio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install -q diffusers transformers accelerate safetensors gradio pillow

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

#---------------------- ***AI IMAGE GENERATOR UI ✨💫***----------------#

In [ ]:
import torch
import gradio as gr
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

# =========================
# Device Setup
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# Load SDXL (High Accuracy)
# =========================
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

# =========================
# Prompt Builder
# =========================
def build_prompt(subject, background, lighting, mood, realism, detail):

    parts = []

    if subject:
        parts.append(f"photorealistic {subject}")

    if background:
        parts.append(f"in {background}")

    if lighting:
        parts.append(f"{lighting} lighting")

    if mood:
        parts.append(mood)

    # realism control
    if realism >= 7:
        parts.append("realistic proportions, natural colors, professional photography")
    elif realism >= 4:
        parts.append("cinematic composition, detailed environment")
    else:
        parts.append("creative composition")

    # detail control
    if detail == "Ultra":
        parts.append("ultra detailed, sharp focus, 8k texture")
    elif detail == "High":
        parts.append("high detail, sharp focus")
    else:
        parts.append("clean composition")

    return ", ".join(parts)

# =========================
# Image Generator
# =========================
def generate_image(prompt, negative, speed, seed):

    if speed == "Fast":
        steps = 25
        cfg = 6.5
    elif speed == "Balanced":
        steps = 35
        cfg = 7.5
    else:
        steps = 45
        cfg = 8.5

    generator = torch.Generator(device=device).manual_seed(int(seed))

    image = pipe(
        prompt=prompt,
        negative_prompt=negative,
        num_inference_steps=steps,
        guidance_scale=cfg,
        generator=generator,
        height=1024,
        width=1024
    ).images[0]

    return image

# =========================
# Attractive UI
# =========================
with gr.Blocks(theme=gr.themes.Soft(), title="AI Image Generator — Precision SDXL") as demo:

    gr.Markdown("# 🎨 AI Image Generator")
    gr.Markdown("Using Gradio UI ✨")
    #Structured prompting with SDXL photorealism

    with gr.Row():

        with gr.Column(scale=1):

            subject = gr.Textbox(
                label="Main Subject",
                placeholder="young football player"
            )

            background = gr.Dropdown(
                [
                    "green football field",
                    "modern city skyline",
                    "mountain landscape",
                    "beach at sunset",
                    "studio background",
                    "cyberpunk city",
                    "royal palace interior",
                    "forest nature scene"
                ],
                label="Background / Environment"
            )

            lighting = gr.Dropdown(
                ["soft natural", "golden hour", "studio", "dramatic", "misty"],
                value="soft natural",
                label="Lighting"
            )

            mood = gr.Dropdown(
                [
                    "cinematic realism",
                    "natural realism",
                    "professional photography",
                    "dramatic atmosphere",
                    "minimalist clean style",
                    "epic cinematic"
                ],
                value="cinematic realism",
                label="Mood / Style"
            )

            realism = gr.Slider(0, 10, value=8, label="Realism → Creativity")
            detail = gr.Dropdown(["Low", "High", "Ultra"], value="Ultra", label="Detail Level")

            negative = gr.Textbox(
                label="Negative Prompt",
                value="extra fingers, deformed hands, bad anatomy, blurry, distorted face"
            )

            speed = gr.Radio(
                ["Fast", "Balanced", "Ultra Quality"],
                value="Balanced",
                label="Generation Mode"
            )

            seed = gr.Slider(0, 999999, value=1234, label="Seed")

            generate_btn = gr.Button("✨ Generate Image", variant="primary")

        with gr.Column(scale=1):

            prompt_output = gr.Textbox(label="Generated Prompt")
            image_output = gr.Image(label="Result")
            download_btn = gr.File(label="Download Image")

    def run(subject, background, lighting, mood, realism, detail,
            negative, speed, seed):

        prompt = build_prompt(subject, background, lighting, mood, realism, detail)
        image = generate_image(prompt, negative, speed, seed)

        path = "generated_image.png"
        image.save(path)

        return prompt, image, path

    generate_btn.click(
        run,
        inputs=[
            subject, background, lighting, mood, realism, detail,
            negative, speed, seed
        ],
        outputs=[prompt_output, image_output, download_btn]
    )

demo.launch(share=True)